# Exploración de datos · Reto OT Consulting
**Valeria López · A01352599 · 17 de noviembre de 2026**

En este cuaderno exploro los 8 Exceles que nos entregó OT Consulting, archivo por archivo y columna por columna. Las respuestas a lo que voy encontrando las dejo en celdas de texto como esta, justo debajo del código que las respalda.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
pd.set_option('display.max_columns', 80); pd.set_option('display.width', 200)
RUTA = Path('datos')   # carpeta donde están los 8 .xlsx
archivos = sorted(RUTA.glob('*.xlsx'))
dfs = {a.stem: pd.read_excel(a) for a in archivos}
pd.DataFrame({'archivo':list(dfs), 'filas':[d.shape[0] for d in dfs.values()], 'columnas':[d.shape[1] for d in dfs.values()]})

,archivo,filas,columnas
0,Asignaciones_Colaboradores,31,11
1,Catalogo_Activos,123,11
2,Catalogo_Clientes,12,13
3,Catalogo_Colaboradores,107,45
4,Catalogo_Departamentos,6,7
5,Catalogo_Habilidades,71,6
6,Catalogo_Roles,55,6
7,Matriz_Habilidades,54,77


## 1. Perfil general de cada archivo
Para cada columna calculo el tipo, cuántos vacíos tiene, cuántos valores distintos y los más comunes. Además cuento los **vacíos disfrazados**: textos como "Pendiente", "PENDIENTE", "N/A", fechas de relleno como 1990-01-01 y valores de prueba como 999999999. Pandas no los marca como nulos, pero en la práctica son datos faltantes.

In [2]:
RELLENO = {'pendiente','n/a','na','sin asignar','conocida','xxxxxxxxxx','xxxxxxxxx','9999999999','999999999','colaborador@otconsulting.com.mx','1990-01-01','pendiente llamar','no lo envio'}
def es_relleno(v):
    if pd.isna(v): return False
    s = str(v).strip().lower()
    return s in RELLENO or s.startswith('1990-01-01')

def perfil(df):
    filas=[]
    for c in df.columns:
        s=df[c]
        filas.append({'columna':c,'tipo':str(s.dtype),'nulos':int(s.isna().sum()),
            'relleno':int(s.map(es_relleno).sum()),
            '%_faltante':round(100*(s.isna().sum()+s.map(es_relleno).sum())/len(s),1),
            'distintos':s.nunique(),'top':dict(s.astype(str).value_counts().head(3))})
    return pd.DataFrame(filas)

perfiles = {n: perfil(d) for n,d in dfs.items()}
perfiles['Catalogo_Colaboradores']

,columna,tipo,nulos,relleno,%_faltante,distintos,top
0,ID,str,0,0,0.0,107,"{'OTC003': 1, 'OTC004': 1, 'OTC005': 1}"
1,Nombre Completo,str,0,0,0.0,107,"{'ABRAHAM PEREZ GIL': 1, 'ADRIANA PEREZ NAVARR..."
2,Género,str,0,0,0.0,2,"{'Masculino': 77, 'Femenino': 30}"
3,Fecha Nacimiento,str,1,0,0.9,104,"{'1986-02-24': 2, '1976-08-08': 2, '1985-07-18..."
4,Teléfono,int64,0,107,100.0,1,{'999999999': 107}
5,Correo Electrónico,str,0,107,100.0,1,{'colaborador@otconsulting.com.mx': 107}
6,Ciudad,str,0,0,0.0,25,"{'Monterrey': 50, 'Edo de México': 12, 'CDMX':..."
7,Domicilio,str,0,107,100.0,1,{'conocida': 107}
8,RFC,str,0,107,100.0,1,{'XXXXXXXXXX': 107}
9,CURP,str,0,107,100.0,1,{'XXXXXXXXX': 107}


De las 45 columnas de Colaboradores, varias tienen más del 85% de faltantes reales (Beneficiario, Parentesco, Alergias, Notas, Departamento, Contrato, Infonavit, Manutención). Teléfono, correo, RFC, CURP, NSS y domicilio traen el mismo valor de relleno en los 107 registros. Supongo que el socio los anonimizó, pero igual el modelo tiene que validar su formato.

In [3]:
perfiles['Asignaciones_Colaboradores']

,columna,tipo,nulos,relleno,%_faltante,distintos,top
0,ID Colaborador,str,0,0,0.0,30,"{'OTC095': 2, 'OTC004': 1, 'OTC005': 1}"
1,Nombre Colaborador,str,0,0,0.0,30,"{'SOFÍA EMIRA PEREZ MORALES': 2, 'ADRIANA PERE..."
2,Nómina/Rol Interno,str,0,14,45.2,11,"{'PENDIENTE': 14, 'CONSULTOR FUNCIONAL ORACLE ..."
3,Estatus Colaborador,str,0,0,0.0,1,{'Activo': 31}
4,Cliente,str,0,0,0.0,3,"{'METALSA': 23, 'Allegion': 5, 'Grupo Daltile ..."
5,Proyecto,str,0,0,0.0,4,"{'Asignación indefinida': 18, 'Soporte AMS Man..."
6,Rol Asignado,str,0,29,93.5,2,"{'PENDIENTE': 29, 'CONSULTOR FUNCIONAL ORACLE ..."
7,Tipo de Asignación,str,0,0,0.0,2,"{'Proyecto': 16, 'Asignación': 15}"
8,Estatus Asignación,str,0,0,0.0,1,{'En Curso': 31}
9,Fecha Inicio,str,0,21,67.7,4,"{'1990-01-01': 21, '2026-10-13': 5, '2026-01-0..."


In [4]:
perfiles['Catalogo_Clientes']

,columna,tipo,nulos,relleno,%_faltante,distintos,top
0,Cód Cliente,str,0,0,0.0,8,"{'C0001': 3, 'C0002': 3, 'C0003': 1}"
1,Cliente,str,0,0,0.0,8,"{'METALSA': 3, 'Grupo Daltile ': 3, 'Allegion'..."
2,Industria,str,0,0,0.0,5,"{'Manufactura': 6, 'Automotriz': 3, 'Manufactu..."
3,RFC,str,8,0,66.7,4,"{'PSM991202C2A': 1, 'RMO100429T23 ': 1, 'FAN87..."
4,Contacto Principal,str,0,0,0.0,8,"{'Hernán Macías': 3, 'Luis Mora': 3, 'Jose Lui..."
5,Email Contacto,str,3,0,25.0,7,"{'luis.mora@daltile.com.mx': 3, 'jose.barrios@..."
6,Estatus Cliente,str,0,0,0.0,1,{'Activo': 12}
7,Proyecto,str,0,0,0.0,12,"{'AMS Manufactura': 1, 'Soporte AMS Manufacta'..."
8,Fecha Inicio,str,6,0,50.0,6,"{'2026-07-15': 1, '2026-10-13': 1, '2025-06-01..."
9,Fecha Fin,str,9,0,75.0,3,"{'2026-09-23': 1, '2028-10-30': 1, '2026-10-30..."


In [5]:
perfiles['Catalogo_Activos']

,columna,tipo,nulos,relleno,%_faltante,distintos,top
0,Cód Activo,str,0,0,0.0,123,"{'A0001': 1, 'A0002': 1, 'A0003': 1}"
1,Descripción,str,0,0,0.0,2,"{'Computadora': 122, 'Monitor': 1}"
2,Marca,str,0,0,0.0,12,"{'DELL': 54, 'HP': 23, 'LENOVO': 17}"
3,Modelo,str,1,0,0.8,78,"{'Inspiron Business 3520': 20, 'V14 G2 ITL': 5..."
4,Serie,str,0,1,0.8,123,"{'PF3WHKFD': 1, '5CG1502PSD': 1, '9P1GLQ2': 1}"
5,Tipo,str,0,0,0.0,2,"{'Laptop': 122, 'Monitor': 1}"
6,Estatus,str,0,0,0.0,1,{'Activo': 123}
7,Costo,float64,0,0,0.0,31,"{'0.0': 70, '7999.0': 5, '11699.0': 4}"
8,Colaborador Asignado,str,0,32,26.0,74,"{'Sin Asignar': 32, 'CLAUDIO FABIAN PEREZ ANGU..."
9,Fecha Inicio,str,32,35,54.5,48,"{'1990-01-01': 35, '2025-08-25': 2, '2024-10-1..."


In [6]:
perfiles['Catalogo_Roles']

,columna,tipo,nulos,relleno,%_faltante,distintos,top
0,Código,str,0,0,0.0,55,"{'R0087': 1, 'R0088': 1, 'R0089': 1}"
1,Nombre del Rol,str,0,1,1.8,55,"{'DIRECTOR GENERAL': 1, 'CONSULTOR FUNCIONAL O..."
2,Nivel,str,0,0,0.0,7,"{'Senior': 16, 'Junior': 15, 'Analista': 14}"
3,Departamento,str,2,0,3.6,5,"{'Operaciones y Soporte': 45, 'Dirección Gener..."
4,Descripción,str,0,1,1.8,55,"{'Director General': 1, 'CONSULTOR FUNCIONAL O..."
5,Estatus,str,0,0,0.0,1,{'Activo': 55}


In [7]:
perfiles['Catalogo_Departamentos']

,columna,tipo,nulos,relleno,%_faltante,distintos,top
0,objetivo,str,0,0,0.0,6,"{'Conseguir nuevos clientes, dar seguimiento a..."
1,codigo,str,0,0,0.0,6,"{'D0001': 1, 'D0002': 1, 'D0003': 1}"
2,nombre,str,0,0,0.0,6,"{'Ventas': 1, 'Administración y Finanzas': 1, ..."
3,centroCostos,float64,6,0,100.0,0,{}
4,uid,int64,0,0,0.0,6,"{'1789623822518': 1, '1789623978382': 1, '1789..."
5,responsable,float64,6,0,100.0,0,{}
6,estatus,str,0,0,0.0,1,{'Activo': 6}


In [8]:
perfiles['Catalogo_Habilidades']

,columna,tipo,nulos,relleno,%_faltante,distintos,top
0,uid,int64,0,0,0.0,71,"{'1789777237350': 1, '1789777249241': 1, '1789..."
1,descripcion,str,0,0,0.0,58,"{'GL': 2, 'SLA': 2, 'CE': 2}"
2,estatus,str,0,0,0.0,1,{'Activo': 71}
3,tecnologia,str,0,0,0.0,6,"{'Oracle eBS': 30, 'Plataformas Tecnológicas':..."
4,codigo,str,0,0,0.0,71,"{'H0002': 1, 'H0003': 1, 'H0004': 1}"
5,tipo,str,0,0,0.0,2,"{'HARD': 67, 'SOFT': 4}"


In [9]:
m = perfiles['Matriz_Habilidades']
m[m.columna.str.match(r'^[HS]\d{4}')].sort_values('%_faltante', ascending=False).head(15)

,columna,tipo,nulos,relleno,%_faltante,distintos,top
76,S0005 - Manejo de Conflictos,float64,54,0,100.0,0,{}
75,S0004 - Empatia,float64,54,0,100.0,0,{}
32,H0028 - TALEO,str,45,0,83.3,2,"{'Básico': 7, 'Intermedio': 2}"
66,H0062 - FUSE,str,44,0,81.5,2,"{'Básico': 8, 'Intermedio': 2}"
50,H0046 - BC,str,43,0,79.6,3,"{'Básico': 7, 'Intermedio': 2, 'Experto': 2}"
27,H0023 - OPM,str,42,0,77.8,2,"{'Básico': 9, 'Intermedio': 3}"
33,H0029 - PRJ,str,42,0,77.8,4,"{'Básico': 8, 'Intermedio': 2, 'Avanzado': 1}"
73,S0002 - CERTIFICADO ORACLE,str,42,0,77.8,1,{'Experto': 12}
34,H0030 - eAM,str,40,0,74.1,4,"{'Básico': 6, 'Intermedio': 6, 'Avanzado': 1}"
49,H0045 - MFG,str,40,0,74.1,3,"{'Básico': 8, 'Intermedio': 5, 'Avanzado': 1}"


## 2. Resumen de faltantes por archivo

In [10]:
resumen = pd.DataFrame([{'archivo':n,'celdas':d.size,
   'nulos':int(d.isna().sum().sum()),
   'relleno':int(d.map(es_relleno).sum().sum())} for n,d in dfs.items()])
resumen['%_faltante'] = (100*(resumen.nulos+resumen.relleno)/resumen.celdas).round(1)
resumen.sort_values('%_faltante', ascending=False)

,archivo,celdas,nulos,relleno,%_faltante
7,Matriz_Habilidades,4158,2356,82,58.6
3,Catalogo_Colaboradores,4815,427,1625,42.6
4,Catalogo_Departamentos,42,12,0,28.6
2,Catalogo_Clientes,156,38,0,24.4
0,Asignaciones_Colaboradores,341,0,64,18.8
1,Catalogo_Activos,1353,65,68,9.8
6,Catalogo_Roles,330,2,2,1.2
5,Catalogo_Habilidades,426,0,0,0.0


## 3. Integridad entre archivos
Aquí reviso si las tablas realmente se pueden conectar: que los IDs existan del otro lado, que los nombres coincidan y que no haya registros huérfanos.

In [11]:
col = dfs['Catalogo_Colaboradores']; asg = dfs['Asignaciones_Colaboradores']; act = dfs['Catalogo_Activos']
cli = dfs['Catalogo_Clientes']; rol = dfs['Catalogo_Roles']; hab = dfs['Catalogo_Habilidades']; mat = dfs['Matriz_Habilidades']
estatus = col.set_index('ID')['Estatus']; por_nombre = col.set_index('Nombre Completo')['Estatus']
activos = col[col.Estatus=='Activo']
asig = act[act['Colaborador Asignado']!='Sin Asignar']
print('Colaboradores activos:', len(activos), '| bajas:', (col.Estatus=='Baja').sum())
print('Activos sin ninguna asignación:', (~activos.ID.isin(asg['ID Colaborador'])).sum())
print('Asignaciones a "Asignación indefinida":', (asg.Proyecto=='Asignación indefinida').sum())
print('Asignaciones con Rol Asignado PENDIENTE:', (asg['Rol Asignado']=='PENDIENTE').sum(), 'de', len(asg))
print('Asignaciones con fecha inicio 1990-01-01:', asg['Fecha Inicio'].astype(str).str.startswith('1990').sum())
print('Asignaciones con fecha fin "Indefinido":', (asg['Fecha Fin'].astype(str)=='Indefinido').sum())
print('Equipos asignados a colaboradores dados de baja:', (asig['Colaborador Asignado'].map(por_nombre)=='Baja').sum())
print('Equipos con costo 0:', (act.Costo==0).sum(), 'de', len(act))
print('Activos que no están en la Matriz de habilidades:', (~activos.ID.isin(mat['ID Colaborador'])).sum())
print('Puestos en Colaboradores que no existen en Roles:', set(col.Puesto)-set(rol['Nombre del Rol']))

Colaboradores activos: 75 | bajas: 32
Activos sin ninguna asignación: 45
Asignaciones a "Asignación indefinida": 18
Asignaciones con Rol Asignado PENDIENTE: 29 de 31
Asignaciones con fecha inicio 1990-01-01: 21
Asignaciones con fecha fin "Indefinido": 24
Equipos asignados a colaboradores dados de baja: 17
Equipos con costo 0: 70 de 123
Activos que no están en la Matriz de habilidades: 28
Puestos en Colaboradores que no existen en Roles: set()


In [12]:
# Mismo cliente o proyecto escrito distinto (espacios y typos)
for c in ['Cliente','Industria','Proyecto','PM OTC']:
    print(c, '->', sorted(cli[c].dropna().astype(str).unique()))
print('Marcas:', act.Marca.value_counts().to_dict())

Cliente -> ['Allegion', 'CITI', 'Fanosa', 'Grupo Daltile ', 'METALSA', 'Novocast', 'Parker Servicios de México ', 'Rent A Center (RAC Mèxico Operaciones)']
Industria -> ['Automotriz', 'Consultoría', 'Manufactura', 'Manufactura ', 'Rentas de equipo ']
Proyecto -> ['AMS Manufactura', 'Asignación EBS temporal (3 meses)', 'Asignación Funcional-Técnico (4 meses)', 'Asignación indefinida', 'Desarrollo de Indicadores ', 'Hosting eBS', 'Marazzi', 'Migración eBs ', 'SHINE - Asiganci´ñon de recursos de apoyo', 'Soporte AMS Manufacta', 'Soporte DBA', 'Soporte Oracle Cloud']
PM OTC -> ['JUAN JOSÉ CORONADO BRAVO', 'José Francisco Mendoza Bazán', 'Juan José Coronado Bravo']
Marcas: {'DELL': 54, 'HP': 23, 'LENOVO': 17, 'ACER': 8, 'Lenovo': 5, 'DELL ': 4, 'HP ': 3, 'Dell': 3, 'LENOVO ': 2, 'ASUS': 2, 'Dell ': 1, 'Acer': 1}


In [13]:
# Habilidades: la misma descripción en eBS y en Cloud, y columnas de la matriz sin evaluar
dup = hab[hab.descripcion.duplicated(keep=False)].sort_values('descripcion')[['codigo','descripcion','tecnologia']]
print('Habilidades repetidas entre tecnologías:', dup.descripcion.nunique())
sk = [c for c in mat.columns if c[:1] in 'HS' and c[1:5].isdigit()]
print('Columnas de habilidad nunca evaluadas:', [c for c in sk if mat[c].isna().all()])

Habilidades repetidas entre tecnologías: 13
Columnas de habilidad nunca evaluadas: ['S0004 - Empatia ', 'S0005 - Manejo de Conflictos ']


## 4. Nómina
Comprobé cómo se forman los sueldos y dónde están los ceros.

In [14]:
S = ['Sueldo Neto IMSS','Sueldo Asimilados/Sindicato','Sueldo Complementos','Sueldo Neto Total','Comisión Nómina','Impuestos IMSS','Sueldo Total Integrado']
neto = col[S[0]]+col[S[1]]+col[S[2]]
integ = col['Sueldo Neto Total']+col['Comisión Nómina']+col['Impuestos IMSS']
print('Neto Total = IMSS + Asimilados + Complementos en', ((neto-col['Sueldo Neto Total']).abs()<1).sum(), 'de', len(col))
print('Integrado = Neto Total + Comisión + Impuestos IMSS en', ((integ-col['Sueldo Total Integrado']).abs()<1).sum(), 'de', len(col))
ceros = col[col['Sueldo Neto Total']==0]
print('Sueldo total en 0:', len(ceros), ceros.Estatus.value_counts().to_dict())
print('Bajas que conservan sueldo:', ((col.Estatus=='Baja')&(col['Sueldo Neto Total']>0)).sum())
r = col[col['Sueldo Neto IMSS']>0]
print('Impuestos IMSS / Neto IMSS (mediana, máximo):', round((r['Impuestos IMSS']/r['Sueldo Neto IMSS']).median(),2), round((r['Impuestos IMSS']/r['Sueldo Neto IMSS']).max(),2))
a = activos[activos['Sueldo Neto Total']>0]
print('Mediana de la parte IMSS sobre el neto (activos):', round((a['Sueldo Neto IMSS']/a['Sueldo Neto Total']).median(),2))
print('Costo integrado mensual de activos:', round(a['Sueldo Total Integrado'].sum()))
banca = a[~a.ID.isin(asg['ID Colaborador']) & a.Puesto.str.contains('CONSULTOR')]
print('Consultores sin asignación:', len(banca), '| costo mensual:', round(banca['Sueldo Total Integrado'].sum()))
indef = a[a.ID.isin(asg[asg.Proyecto=='Asignación indefinida']['ID Colaborador'])]
print('En "Asignación indefinida":', len(indef), '| costo mensual:', round(indef['Sueldo Total Integrado'].sum()))

Neto Total = IMSS + Asimilados + Complementos en 107 de 107
Integrado = Neto Total + Comisión + Impuestos IMSS en 107 de 107
Sueldo total en 0: 25 {'Baja': 20, 'Activo': 5}
Bajas que conservan sueldo: 12
Impuestos IMSS / Neto IMSS (mediana, máximo): 0.38 6.54
Mediana de la parte IMSS sobre el neto (activos): 0.21
Costo integrado mensual de activos: 3918864
Consultores sin asignación: 21 | costo mensual: 1086301
En "Asignación indefinida": 18 | costo mensual: 1319330


In [15]:
pd.crosstab(col['Tipo Nómina'].fillna('(vacío)'), col.Estatus)

Estatus,Activo,Baja
Tipo Nómina,,
(vacío),0,6
ADMINISTRATIVA,2,0
EXTRANJERO,0,1
IRAZU/ASIMILADOS,1,0
MAGNESON-SINDICATO,10,10
MODALIDAD 40-SMR,3,0
OTC 100%,14,4
OTC-SINDICATO (SMR),10,0
SMR,29,7


## 5. Conclusión de la exploración
Los datos alcanzan para armar el catálogo de personas, roles, clientes y equipo, pero no para lo que más le importa al socio: saber quién está disponible, cuándo se libera y cuánto cuesta. Las asignaciones no tienen fechas reales ni rol, los proyectos no tienen fases y la matriz de habilidades no cubre a 28 de los activos. Todo esto lo detallo en el documento del reto, con el modelo de datos que propongo para corregirlo.